# Data Cleaning

This notebook performs the cleaning and standardization of the LendingClub dataset.

### Phase 1: Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np

# Load the raw data
raw_data_path = '../data/raw/lending_club_loan_two.csv'
df = pd.read_csv(raw_data_path)

# Integrity Check
df_copy = df.copy()
print(f"Original dataset shape: {df.shape}")

### Phase 2: Schema Standardization

In [ ]:
# 2.1 Column Renaming
df.rename(columns={
    'loan_amnt': 'loan_amount',
    'int_rate': 'interest_rate',
    'installment': 'monthly_installment',
    'annual_inc': 'annual_income',
    'dti': 'debt_to_income_ratio',
    'open_acc': 'number_of_open_accounts',
    'pub_rec': 'public_records_derogatory',
    'revol_bal': 'revolving_balance',
    'revol_util': 'revolving_line_utilization',
    'total_acc': 'total_credit_lines',
    'pub_rec_bankruptcies': 'public_record_bankruptcies',
    'issue_d': 'issue_date',
    'earliest_cr_line': 'earliest_credit_line_opened',
    'term': 'loan_term',
    'grade': 'loan_grade',
    'sub_grade': 'loan_sub_grade',
    'home_ownership': 'home_ownership_status',
    'verification_status': 'income_verification_status',
    'loan_status': 'loan_repayment_status',
    'purpose': 'loan_purpose',
    'title': 'loan_title_description',
    'zip_code': 'zip_code_prefix',
    'addr_state': 'borrower_state_address',
    'initial_list_status': 'initial_listing_status',
    'application_type': 'loan_application_type',
    'address': 'borrower_full_address',
    'log_income': 'log_annual_income',
    'mort_acc':'number_of_mortgage_accounts'
}, inplace=True)

# 2.2 Verification
print("Columns after renaming:")
print(df.columns)

### Phase 3: Structural Cleaning

In [ ]:
# 3.1 Drop Redundant Columns
df.drop(['emp_title', 'emp_length', 'number_of_mortgage_accounts'], axis=1, inplace=True)

# 3.2 Handle Duplicates
duplicates_count = df.duplicated().sum()
print(f"Number of duplicates found: {duplicates_count}")
df.drop_duplicates(inplace=True)

### Phase 4: Missing Value Treatment

In [ ]:
# 4.1 Numeric Imputation (Median)
df['revolving_line_utilization'].fillna(df['revolving_line_utilization'].median(), inplace=True)
df['public_record_bankruptcies'].fillna(df['public_record_bankruptcies'].median(), inplace=True)

# 4.2 Categorical Imputation
df['loan_title_description'].fillna("Unknown", inplace=True)

# 4.3 Final Removal of remaining nulls
df.dropna(inplace=True)

print("Null values after treatment:")
print(df.isnull().sum())

### Phase 5: Data Type & Outlier Cleaning

In [ ]:
# 5.1 Type Casting
df['loan_amount'] = df['loan_amount'].astype(float)
df['annual_income'] = df['annual_income'].astype(float)

# 5.2 DateTime Parsing
df['issue_date'] = pd.to_datetime(df['issue_date'], format='%b-%Y')

# 5.3 String Trimming
df['loan_term'] = df['loan_term'].str.strip()

# 5.4 Outlier Filtering (Annual Income < 99th percentile)
df = df[df['annual_income'] < df['annual_income'].quantile(0.99)]

print("Final Data Types:")
print(df.dtypes)

### Phase 6: Quality Audit & Export

In [ ]:
# Final Statistics and Validation
print("--- Final Dataset Info ---")
df.info()

print("\n--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Descriptive Statistics ---")
display(df.describe())

print(f"\nFinal dataset shape: {df.shape}")

# Export Cleaned Data
output_path = '../data/processed/cleaned_club_loan_two.csv'
df.to_csv(output_path, index=False)
print(f"\nCleaned dataset saved successfully to: {output_path}")

### Phase 7: Feature Engineering

In [ ]:
# ------------------------------------------------
# 1. loan_income_ratio
# Meaning: Loan amount compared to the borrower's yearly income.
# Purpose: Shows how large the loan is relative to income.
# Higher value means the borrower is taking a bigger loan burden.
# ------------------------------------------------
df["loan_income_ratio"] = df["loan_amount"] / df["annual_income"]


# ------------------------------------------------
# 2. installment_income_ratio
# Meaning: Monthly installment compared to monthly income.
# Purpose: Measures how much of the borrower's monthly income
# goes toward loan repayment.
# Higher value means higher monthly repayment pressure.
# ------------------------------------------------
df["installment_income_ratio"] = (
    df["monthly_installment"] / (df["annual_income"] / 12)
)


# ------------------------------------------------
# 3. risk_score
# Meaning: Combined risk indicator using DTI, utilization,
# loan-income ratio, and bankruptcy records.
# Purpose: Gives a single numeric score to compare borrower risk.
# Higher score means the borrower may be financially riskier.
# ------------------------------------------------
df["risk_score"] = (
    0.4 * df["debt_to_income_ratio"] +
    0.3 * df["revolving_line_utilization"] +
    0.2 * df["loan_income_ratio"] * 100 +
    0.1 * df["public_record_bankruptcies"].fillna(0) * 10
)


# ------------------------------------------------
# 4. credit_stress_score
# Meaning: Measures credit pressure using DTI, credit utilization,
# and derogatory public records.
# Purpose: Helps identify borrowers under financial stress.
# Higher score means higher credit stress.
# ------------------------------------------------
df["credit_stress_score"] = (
    df["debt_to_income_ratio"] +
    df["revolving_line_utilization"] +
    (df["public_records_derogatory"].fillna(0) * 10)
)


# ------------------------------------------------
# 5. issue_year
# Meaning: Year in which the loan was issued.
# Purpose: Useful for year-wise trend analysis such as
# loan volume, average loan amount, interest rate, or risk over time.
# ------------------------------------------------
df["issue_date"] = pd.to_datetime(
    df["issue_date"].astype(str),
    format="%b-%Y",
    errors="coerce"
)

df["issue_year"] = df["issue_date"].dt.year


# ------------------------------------------------
# 6. income_band
# Meaning: Groups borrowers into income categories.
# Purpose: Makes income-based EDA easier by comparing
# low, middle, and high income borrowers.
# ------------------------------------------------
df["income_band"] = pd.cut(
    df["annual_income"],
    bins=[0, 40000, 100000, np.inf],
    labels=["Low", "Middle", "High"]
)


# ------------------------------------------------
# 7. dti_band
# Meaning: Groups borrowers based on debt-to-income ratio.
# Purpose: Helps compare borrowers with low, medium,
# and high debt burden.
# ------------------------------------------------
df["dti_band"] = pd.cut(
    df["debt_to_income_ratio"],
    bins=[0, 10, 20, np.inf],
    labels=["Low", "Medium", "High"]
)


# ------------------------------------------------
# 8. utilization_band
# Meaning: Groups borrowers based on revolving credit utilization.
# Purpose: Helps compare borrowers with low, medium,
# and high credit usage.
# Higher utilization usually means higher financial pressure.
# ------------------------------------------------
df["utilization_band"] = pd.cut(
    df["revolving_line_utilization"],
    bins=[0, 30, 70, 100],
    labels=["Low", "Medium", "High"]
)


df.head()